## 00_linear_least_squares.py

![Alt text](./МНК1.png)

![Alt text](./МНК2.png)

## Задача 1
Написать в функцию, реализующую
формулы для проведения аппроксимирующей прямой, приведённые выше

## Задача 2
Применить полученную при решении Задачи 1 функцию
к данным из файла second_day/data_files/data.csv.
Отрисовать экспериментальные точки и полученную прямую

## 01_curve_fit.py

In [ ]:
from scipy.optimize import curve_fit

import numpy as np
import matplotlib.pyplot as plt

# Зададим функцию, которая должна приближать
# экспериментальные данные
def func(x, a, b):
    return a * x ** 2 + np.sin(3*x) - b

# Количество рассматриваемых точек
N = 50

# Параметры функции
A = 2.5
B = 3

x = np.linspace(0, 2, N)
y = func(x, A, B)

In [ ]:
# Синтезируем экспериментальные данные,
# добавив к точкам графика случайный шум

# Амплитуда шума
n_amp = 0.1

x_exp = x + n_amp * (np.random.random(N) - 0.5)
y_exp = y + n_amp * (np.random.random(N) - 0.5)

# Пробуем подобрать коэффициенты a и b, чтобы
# аппроксимировать шумные данные
coefs, pcov = curve_fit(func, x_exp, y_exp)

# Погрешность определения каждого коэффициента
coef_error = np.sqrt(np.diag(pcov))

print(coefs, coef_error)

plt.plot(x, y, lw=4.0)
plt.plot(x_exp, y_exp, 'rv', ms=3)

plt.plot(x, func(x, coefs[0], coefs[1]), 'm-.', lw=1.5)
plt.show()

## 02_hard_case.py

In [ ]:
from scipy.optimize import curve_fit

import numpy as np
import matplotlib.pyplot as plt

# Теперь функция будет немного хитрее
def func(x, a, b, c):
    return (-np.sign(x) * a * (-1e-5) * np.log(np.abs(70*x)) 
            + b / 100 * np.cos(c / 1000 * x))

# Количество рассматриваемых точек
N = 120

# Параметры функции
A = -5e4
B = -400
C = 2000

real_coefs = np.array([A, B, C])

x = np.linspace(-2, 2, N)
y = func(x, A, B, C)

# Амплитуда шума теперь выше
n_amp = 0.3

x_exp = x + n_amp * (np.random.random(N) - 0.5)
y_exp = y + n_amp * (np.random.random(N) - 0.5)

In [ ]:
# Пробуем аппроксимировать

# Так получаются совсем неправильные значения
coefs, pcov = curve_fit(func, x_exp, y_exp)

# Но обычно мы можем каким-то образом
# оценить диапазон значений параметров
bounds = ((-9e4, -1_000, 1_000),
          (-1e4, -100.0, 4_000))
# или их приближённые значения
p0 = np.array([-3.94e4, -267, 2200])

# Тогда можно использовать эти дополнительные
# данные для подбора коэффициентов
# (можно использовать или диапазон, 
# или приближённые значения, одновременно не
# получится)
# coefs, pcov = curve_fit(func, x_exp, y_exp, 
#                         bounds=bounds)

# или
# coefs, pcov = curve_fit(func, x_exp, y_exp, 
#                         p0=p0)

# Погрешность определения каждого коэффициента
coef_error = np.sqrt(np.diag(pcov))


# Обрежем точность выводимых на экран чисел
# до 2-х знаков после запятой
np.set_printoptions(precision=2)
print('Истинные значения: ', real_coefs)
print('Подобранные значения:', coefs)
print('Оценка абс. погрешности:', coef_error)

# Выведем также относительную 
# погрешность относительно реальных коэффициентов
print('Реальная отн. погрешность, %: ',
      (coefs - real_coefs) / real_coefs * 100)

plt.plot(x, y, lw=4.0)
plt.plot(x_exp, y_exp, 'rv', ms=3)

plt.plot(x, func(x, coefs[0], coefs[1], coefs[2]), 'm-.', lw=1.5)
plt.show()

## Задачи 3, 4, 5
С помощью любой доступной вам LLM решить одну из следующих задач на выбор:

3: По данным в файле third_day/data/Energies.xml, 
содержащем измеренные значения энергии излучённых
кобальтом гамма-квантов, построить гистограмму и
плотность распределения частиц по энергиям на 
одном графике.

4: Вам даны результаты расчета гидродинамики в пористой среде, образце горной породы. Это упрощенная модель, в которой сложная внутренняя структура представлена относительно простой моделью: поры в породе - вершины графа, каналы между ними - ребра. Размер вершины и толщина ребер показывают размеры соответствующих пор и каналов. Цвет вершины - давление жидкости в ней. Попробуйте с помощью matplotlib визуализировать результаты расчета.

Расшифровка формата.

Файл third_day/data/1_000_pressures_2D_0.csv хранит информацию о порах (вершинах графа).
По столбцам:
координата(х) координата(y) координата(z) давление радиус | (два числа для отладки, вам они не нужны)
Обратите внимание, что первая и последняя пора являются необычными. Это фантомные узлы, которые задают граничные условия и имеют очень большой размер в сравнении с остальными. Подумайте, как лучше их отобразить. 

Файл third_day/data/1_000_edges_2D_0.csv хранит информацию о каналах (ребрах графа).
По столбцам:
вершина1 вершина2 радиус
Нумерация вершин с единицы в соответствии с порядком в первом файле. 

У кого получится сделать визуализацию, потом подкинем побольше файлов того же формата.

5: Это довольно сложная задача, уровня сильного второго курса. Но попробуйте - вдруг с помощью нейронок у вас получится справиться с ней уже сейчас. Запрашивайте у нейронок объяснение терминов, просите иллюстрировать концепции.

Существует большой набор методов обработки данных, в которых сложная функция раскладывается на набор более простых с определенными коэффициентами. Скоро вы познакомитесь с рядами Тейлора, а потом и рядами Фурье. В этой задаче вам нужно воспользоваться более сложным, но более красочным рядом функций - сферическими гармониками. 

Предположим, что у вас есть какая-то сложная замкнутая поверхность, заданная сеткой, и вы можете выбрать точку внутри, из которой каждая точка поверхности видна без пересечений это поверхности. Есть теорема, которая доказывает, что вы можете приблизить эту сложную геометрию с любой точностью, если возьмете достаточное количество сферических гармоник и сложите их с правильными коэффициентами. 

Попробуйте это и сделать, пример в видео. Сетки можно брать в формате stl. Одна лежит по второй ссылке, но их в интернете много для 3D-принтеров, можете попробовать разные.
https://github.com/DolgushevVN/spherical-harmonic-expansion/blob/master/anim.mp4
https://github.com/DolgushevVN/spherical-harmonic-expansion/blob/master/4.stl
 
Самые хитрые и внимательные могут найти готовое решение этой задачи, но попробуйте все же разобраться сами и выбить из нейронки приличный код, который можно немного допилить руками и заставить хорошо работать.

## Решение задачи 1

<summary> Нажмите, чтобы показать решение </summary>

<details>

```python
import numpy as np

def lin_ls(x, y):
    """
        Возвращает коэффициенты b, k и их погрешности s_b, s_k при
    линейной аппроксимации y = k * x + b методом МНК по введённым точкам (х, у).

    Parameters
    ----------
    x : numpy.ndarray
        Numpy массив х координат.

    y : numpy.ndarray
        Numpy массив у координат.

    Returns
    -------
    out : tuple
        (k, s_k, b, s_b)
    """
    xy = np.mean(x * y)
    x1y = np.mean(x) * np.mean(y)
    x2 = np.mean(x * x)
    x12 = np.mean(x) ** 2
    y2 = np.mean(y * y)
    y12 = np.mean(y) ** 2
    k = (xy - x1y) / (x2 - x12)
    b = np.mean(y) - k * np.mean(x)
    s_k = np.sqrt(1 / x.size) * np.sqrt((y2 - y12) / (x2 - x12) - k ** 2)
    s_b = s_k * np.sqrt(x2 - x12)
    return k, s_k, b, s_b
```

</details>